In [1]:
import os
import glob
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import cv2
import random
from PIL import Image, ImageEnhance, ImageOps
from segment_anything import sam_model_registry, SamPredictor
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_fscore_support, confusion_matrix
)
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import math

In [2]:
SEED = 42
oiriginal_img_dir = Path("../matchcut_0419")   # 원본 데이터셋 경로
augmented_img_dir = Path("./augImages_0527")   # 출력 경로 (색감 증강 포함)
augmented_emb_dir = Path("./augSAMemb_0527")
model_ckpt_path = Path("./compositionEncoder_model_0527.pth")

device = "cuda" if torch.cuda.is_available() else "cpu"

## 데이터 증강
- rotate, flip, color, grayscale 적용
- 증강본 5개 제작 (원본 x **6**)

In [3]:
def crop_around_center(img: Image.Image, angle: float, orig_w: int, orig_h: int) -> Image.Image:
    """
    회전 방향이나 이미지 비율 분기점 오류 없이,
    어떤 각도에서도 검은 여백(2개의 직각삼각형 포함)을 완벽히 제거하는 보수적 크롭 공식
    """
    angle_rad = math.radians(abs(angle))
    cos_a = math.cos(angle_rad)
    sin_a = math.sin(angle_rad)
    
    # 1:1 정방형 이미지 또는 일반 이미지에서 검은 삼각형을 완전히 피하는 
    # 가장 안전한 가로/세로 내부 크기 계산 (수학적으로 가장 좁은 범위를 채택)
    # 두 가지 경우의 수 중 항상 '더 안쪽으로 잘라내는' 값을 선택하여 꼼수를 차단합니다.
    w1 = (orig_w * cos_a - orig_h * sin_a) / (cos_a**2 - sin_a**2)
    h1 = (orig_h - w1 * sin_a) / cos_a
    
    w2 = (orig_w - orig_h * sin_a) / cos_a
    h2 = (orig_h * cos_a - orig_w * sin_a) / (cos_a**2 - sin_a**2)
    
    # 양수이면서 더 안전하게 작게 잘라내는 크기를 선택
    new_w = min(abs(w1), abs(w2))
    new_h = min(abs(h1), abs(h2))
    
    # 만약 각도가 0도에 가깝거나 공식이 범위를 벗어나면 원본 크기 대입
    if cos_a == sin_a:
        new_w, new_h = orig_w, orig_h

    # 마지막으로 픽셀 오차 방지를 위한 안전 마진 추가
    new_w = max(int(new_w) - 6, 1)
    new_h = max(int(new_h) - 6, 1)
    
    # 현재 확장된 이미지의 중심을 기준으로 크롭
    rot_w, rot_h = img.size
    cx, cy = rot_w / 2, rot_h / 2
    
    left = int(cx - new_w / 2)
    top = int(cy - new_h / 2)
    right = int(cx + new_w / 2)
    bottom = int(cy + new_h / 2)
    
    return img.crop((left, top, right, bottom))

def load_pair(idx: str, src: Path):
    img_a = Image.open(src / f"{idx}_a.jpg").convert("RGB")
    img_b = Image.open(src / f"{idx}_b.jpg").convert("RGB")
    if img_a.size != img_b.size:
        img_b = img_b.resize(img_a.size, Image.LANCZOS)
    
    # [조건 0] 원본 이미지 1:1 비율로 변경 (가장 긴 변의 길이에 맞춰 강제로 늘림)
    max_side = max(img_a.size)
    img_a = img_a.resize((max_side, max_side), Image.BILINEAR)
    img_b = img_b.resize((max_side, max_side), Image.BILINEAR)
    
    return img_a, img_b

def save_pair(img_a: Image.Image, img_b: Image.Image, dst: Path, prefix: str):
    img_a.save(dst / f"{prefix}_a.jpg", quality=95)
    img_b.save(dst / f"{prefix}_b.jpg", quality=95)

def rotate_and_square_pair(img_a: Image.Image, img_b: Image.Image, angle: float):
    """회전 후 검은 여백을 잘라내고(Crop), 다시 1:1 정방형으로 강제 변형하는 함수"""
    # 원본 크기 저장 (load_pair에서 이미 1:1 정방형 스트레칭이 완료된 크기)
    orig_w, orig_h = img_a.size

    # 1. 회전 (expand=True를 해야 원래 이미지가 잘리지 않고 여백을 포함해 커집니다)
    rot_a = img_a.rotate(angle, expand=True, resample=Image.BICUBIC)
    rot_b = img_b.rotate(angle, expand=True, resample=Image.BICUBIC)
    
    # 2. 회전 후 생긴 주위 검은 사각형(여백) 제거를 위한 크롭 (원본 가로/세로 정보 전달)
    cropped_a = crop_around_center(rot_a, angle, orig_w, orig_h)
    cropped_b = crop_around_center(rot_b, angle, orig_w, orig_h)
    
    # 3. [조건 1~5] 최종 출력물 역시 1:1 비율로 강제 변형
    target_size = (orig_w, orig_h)
    final_a = cropped_a.resize(target_size, Image.BILINEAR)
    final_b = cropped_b.resize(target_size, Image.BILINEAR)
    
    return final_a, final_b

def flip_pair(img_a: Image.Image, img_b: Image.Image, direction):
    return img_a.transpose(direction), img_b.transpose(direction)

def apply_color_jitter(img_a: Image.Image, img_b: Image.Image, rng: random.Random):
    b_val = rng.uniform(0.6, 1.4)
    c_val = rng.uniform(0.6, 1.4)
    s_val = rng.uniform(0.5, 1.5)
    
    def jitter(img):
        img = ImageEnhance.Brightness(img).enhance(b_val)
        img = ImageEnhance.Contrast(img).enhance(c_val)
        img = ImageEnhance.Color(img).enhance(s_val)
        return img
    return jitter(img_a), jitter(img_b)

def apply_grayscale(img_a: Image.Image, img_b: Image.Image):
    return ImageOps.grayscale(img_a).convert("RGB"), ImageOps.grayscale(img_b).convert("RGB")

def augment_dataset(INPUT_DIR, OUTPUT_DIR, SEED=42):
    rng = random.Random(SEED)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    indices = sorted(p.stem.replace("_a", "") for p in INPUT_DIR.glob("*_a.jpg"))
    if not indices: return

    total_pairs = len(indices) * 6
    digits = len(str(total_pairs - 1))

    print(f"총 {len(indices)}쌍 발견 -> {total_pairs}쌍으로 증강 시작 (1:1 강제 비율 및 회전 크롭 적용)\n")

    for orig_idx in indices:
        # 0번 슬롯이 적용된(1:1로 늘려진) 원본 이미지를 베이스로 가져옴
        img_a, img_b = load_pair(orig_idx, INPUT_DIR)

        # 0) 원본 (이미 1:1 강제 스트레칭 완료)
        prefix = str(indices.index(orig_idx) * 6 + 0).zfill(digits)
        save_pair(img_a, img_b, OUTPUT_DIR, prefix)

        # 1) rotate (+ 크롭 + 1:1 복원)
        prefix = str(indices.index(orig_idx) * 6 + 1).zfill(digits)
        rot_a, rot_b = rotate_and_square_pair(img_a, img_b, rng.uniform(-30, 30))
        save_pair(rot_a, rot_b, OUTPUT_DIR, prefix)

        # 2) rotate + flip (LR or TB) (+ 크롭 + 1:1 복원)
        prefix = str(indices.index(orig_idx) * 6 + 2).zfill(digits)
        flip_dir = rng.choice([Image.FLIP_LEFT_RIGHT, Image.FLIP_TOP_BOTTOM])
        rot_a, rot_b = rotate_and_square_pair(img_a, img_b, rng.uniform(-30, 30))
        f_a, f_b = flip_pair(rot_a, rot_b, flip_dir)
        save_pair(f_a, f_b, OUTPUT_DIR, prefix)

        # 3) rotate + flip (another) (+ 크롭 + 1:1 복원)
        prefix = str(indices.index(orig_idx) * 6 + 3).zfill(digits)
        flip_dir_2 = Image.FLIP_TOP_BOTTOM if flip_dir == Image.FLIP_LEFT_RIGHT else Image.FLIP_LEFT_RIGHT
        rot_a, rot_b = rotate_and_square_pair(img_a, img_b, rng.uniform(-30, 30))
        f_a, f_b = flip_pair(rot_a, rot_b, flip_dir_2)
        save_pair(f_a, f_b, OUTPUT_DIR, prefix)

        # 4) rotate + flip + Color Jitter (+ 크롭 + 1:1 복원)
        prefix = str(indices.index(orig_idx) * 6 + 4).zfill(digits)
        rot_a, rot_b = rotate_and_square_pair(img_a, img_b, rng.uniform(-30, 30))
        f_a, f_b = flip_pair(rot_a, rot_b, rng.choice([Image.FLIP_LEFT_RIGHT, Image.FLIP_TOP_BOTTOM]))
        jit_a, jit_b = apply_color_jitter(f_a, f_b, rng)
        save_pair(jit_a, jit_b, OUTPUT_DIR, prefix)

        # 5) rotate + flip + Grayscale (+ 크롭 + 1:1 복원)
        prefix = str(indices.index(orig_idx) * 6 + 5).zfill(digits)
        rot_a, rot_b = rotate_and_square_pair(img_a, img_b, rng.uniform(-30, 30))
        f_a, f_b = flip_pair(rot_a, rot_b, rng.choice([Image.FLIP_LEFT_RIGHT, Image.FLIP_TOP_BOTTOM]))
        gray_a, gray_b = apply_grayscale(f_a, f_b)
        save_pair(gray_a, gray_b, OUTPUT_DIR, prefix)
    
    print(f"\n완료! 총 {total_pairs}쌍 -> '{OUTPUT_DIR}'")

# 실행부 (경로 변수는 환경에 맞게 지정하세요)
# from pathlib import Path
# augment_dataset(Path("./original"), Path("./augmented"))

In [4]:
augment_dataset(oiriginal_img_dir, augmented_img_dir)

총 109쌍 발견 -> 654쌍으로 증강 시작 (1:1 강제 비율 및 회전 크롭 적용)


완료! 총 654쌍 -> 'augImages_0527'


## SAM encoder
- feature map 사전에 추출 후 저장

In [5]:
sam = sam_model_registry["vit_h"](checkpoint="../sam_vit_h_4b8939.pth")
sam.eval() # 평가 모드(학습/평가)
sam.to(device)
predictor = SamPredictor(sam) # * SAM 래퍼 클래스

In [6]:
@torch.no_grad()
def sam_embed(img_path,emb_path):
    img_read = cv2.imread(img_path)
    if img_read is None:
        return None
    img = cv2.cvtColor(img_read, cv2.COLOR_BGR2RGB)
    predictor.set_image(img) # * 해당 이미지로 전처리 자동화
    embed = predictor.get_image_embedding() # * 인코더 결과 뽑기
    torch.save(embed, emb_path/f"{Path(img_path).stem}.pt")

In [7]:
entire_emb = len(glob.glob(f"{augmented_img_dir}/*.jpg"))

augmented_emb_dir.mkdir(exist_ok=True)
for idx, sample in enumerate(augmented_img_dir.iterdir()):
    sam_embed(str(sample),augmented_emb_dir)
    if not (idx+1) % (entire_emb//10):
        process_rate = (idx+1) // (entire_emb//10)*10
        print(f"{process_rate} % 진행...")

aug_emb_files = f'{augmented_emb_dir}/*.pt'
print(f"SAM 임베딩 완성 : {len(glob.glob(aug_emb_files))}개 pt 파일")

10 % 진행...
20 % 진행...
30 % 진행...
40 % 진행...
50 % 진행...
60 % 진행...
70 % 진행...
80 % 진행...
90 % 진행...
100 % 진행...
SAM 임베딩 완성 : 1308개 pt 파일


## 학습

### 데이터 로더
- 전체 데이터 : 원본(218개, 109쌍)
- train : 원본(152개, 76쌍) + 증강(760개, 380쌍) = (912개, 456쌍)
- test : 원본(66개, 33쌍)

In [8]:
N_ori = 109 # 원본 데이터 개수
N_aug = 5   # 증강 데이터 개수
ori_emb = [f"{i:03d}" for i in range(0,(N_aug+1)*N_ori, N_aug+1)]

train_ori, test_ori = train_test_split(ori_emb, test_size=0.3, random_state=SEED)

train_ids = list(train_ori)
for i in train_ori:
    train_ids += [f"{int(i)+k:03d}" for k in range(1,N_aug+1)]
train_ids.sort()

test_ids = list(test_ori)

In [9]:
class MatchCutDataset(Dataset):
    def __init__(self, data_id):
        self.data_id = data_id

    def __len__(self):
        return len(self.data_id)

    def __getitem__(self, idx):
        match_id = self.data_id[idx]
        embed_a  = torch.load(augmented_emb_dir/f'{match_id}_a.pt').float().squeeze(0)
        embed_b  = torch.load(augmented_emb_dir/f'{match_id}_b.pt').float().squeeze(0)
        return embed_a, embed_b, match_id

def collate_fn(batch):
    return (
        torch.stack([b[0] for b in batch]),
        torch.stack([b[1] for b in batch]),
        [b[2] for b in batch]
    )

train_dataset = MatchCutDataset(train_ids)
train_loader  = DataLoader(train_dataset, batch_size=32,
                           shuffle=True, collate_fn=collate_fn,
                           drop_last=True)

print(f'train 배치 수: {len(train_loader)}개/epoch')

ea, eb, mids = next(iter(train_loader))
print(f'embed_a shape: {ea.shape}')
print(f'embed_b shape: {eb.shape}')

train 배치 수: 14개/epoch
embed_a shape: torch.Size([32, 256, 64, 64])
embed_b shape: torch.Size([32, 256, 64, 64])


### 딥러닝 계층
**[SCM]** : Spatial Compression Module
- (B,256,64,64) → (B,1024)
- cnn 4개 구성 

**[PH]** : Projection Head
- (B,1024) → (B,128)
- mlp 구성

**[NT-Xent]**
-simCLR의 손실함수

In [10]:
class SCM(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=1),
            nn.BatchNorm2d(128), nn.ReLU()
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(128, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU()
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.ReLU()
        )
        self.conv4 = nn.Sequential(
            nn.Conv2d(32, 16, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(16), nn.ReLU()
        )
        self.flatten = nn.Flatten()

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        return self.flatten(x)  # (B, 1024)

In [11]:
class PH(nn.Module):
    def __init__(self, input_dim=1024, hidden_dim=512, output_dim=128):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return F.normalize(self.mlp(x), p=2, dim=1)

In [12]:
class CompositionEncoder(nn.Module):
    def __init__(self, embed_dim=128):
        super().__init__()
        self.scm  = SCM()
        self.ph = PH(output_dim=embed_dim)

    def forward(self, x, Train=True):
        h = self.scm(x)
        if Train:
            return self.ph(h)
        return h # down-stream work

In [13]:
def nt_xent_loss(e_a, e_b, temperature=0.5):
    # e_a, e_b: (B, 128) L2 정규화된 임베딩
    # e_a[i] ↔ e_b[i] 가 positive 쌍
    B   = e_a.shape[0]
    z   = torch.cat([e_a, e_b], dim=0)          # (2B, 128)
    sim = torch.mm(z, z.T) / temperature         # (2B, 2B)

    mask = torch.eye(2*B, dtype=bool).to(z.device)
    sim.masked_fill_(mask, float('-inf'))

    labels = torch.cat([
        torch.arange(B, 2*B),
        torch.arange(0,   B)
    ]).to(z.device)

    return F.cross_entropy(sim, labels)

### 학습

In [14]:
EPOCHS      = 600
LR          = 1e-3/2
TEMPERATURE = 0.5

model = CompositionEncoder(embed_dim=128).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

print(f'학습 시작: train {len(train_ids)}쌍, {EPOCHS} epoch')
print('-'*50)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for embed_as, embed_bs, _ in train_loader:
        embed_as = embed_as.to(device)
        embed_bs = embed_bs.to(device)

        e_a  = model(embed_as)
        e_b  = model(embed_bs)
        loss = nt_xent_loss(e_a, e_b, TEMPERATURE)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    if epoch % 20 == 0:
        print(f'Epoch [{epoch:3d}/{EPOCHS}]  Loss: {avg_loss:.4f}')

torch.save(model.state_dict(), model_ckpt_path)
print(f'Epoch [{EPOCHS}/{EPOCHS}]  Loss: {avg_loss:.4f}')
print(f'Model Saved (Loss: {avg_loss:.4f}): {model_ckpt_path}')

학습 시작: train 456쌍, 600 epoch
--------------------------------------------------
Epoch [  0/600]  Loss: 3.4829
Epoch [ 20/600]  Loss: 2.2721
Epoch [ 40/600]  Loss: 2.2439
Epoch [ 60/600]  Loss: 2.2363
Epoch [ 80/600]  Loss: 2.2261
Epoch [100/600]  Loss: 2.2237
Epoch [120/600]  Loss: 2.2192
Epoch [140/600]  Loss: 2.2150
Epoch [160/600]  Loss: 2.2151
Epoch [180/600]  Loss: 2.2128
Epoch [200/600]  Loss: 2.2098
Epoch [220/600]  Loss: 2.2104
Epoch [240/600]  Loss: 2.2083
Epoch [260/600]  Loss: 2.2100
Epoch [280/600]  Loss: 2.2061
Epoch [300/600]  Loss: 2.2061
Epoch [320/600]  Loss: 2.2078
Epoch [340/600]  Loss: 2.2058
Epoch [360/600]  Loss: 2.2046
Epoch [380/600]  Loss: 2.2028
Epoch [400/600]  Loss: 2.2039
Epoch [420/600]  Loss: 2.2026
Epoch [440/600]  Loss: 2.2018
Epoch [460/600]  Loss: 2.2016
Epoch [480/600]  Loss: 2.2021
Epoch [500/600]  Loss: 2.2017
Epoch [520/600]  Loss: 2.2008
Epoch [540/600]  Loss: 2.2001
Epoch [560/600]  Loss: 2.2004
Epoch [580/600]  Loss: 2.1996
Epoch [600/600]  Los

### 평가

In [15]:
model.eval()

test_frames = []
for i in test_ids:
    test_frames += [f"{i}_a", f"{i}_b"]

test_emb = {}
with torch.no_grad():
    for j in test_frames:
        test_data = torch.load(augmented_emb_dir / f'{j}.pt').float().squeeze(0).unsqueeze(0).to(device)
        test_emb[j] = model(test_data, Train=False).squeeze(0).cpu()

print(f'구도 임베딩 추출 완료: {len(test_emb)}개')

구도 임베딩 추출 완료: 66개


In [16]:
all_scores = []
all_labels = []

ks = [1, 5, 10]
recall_results = {k: [] for k in ks}

ox_counts = 0

for name_i in test_frames:
    query_scores = []
    query_labels = []
    
    # 각 쿼리 이미지(name_i)에 대해 다른 모든 이미지(name_j)와 비교
    for name_j in test_frames:
        if name_i == name_j:
            continue
            
        # 유사도 계산 (코사인 유사도)
        sim = F.cosine_similarity(
            test_emb[name_i].unsqueeze(0),
            test_emb[name_j].unsqueeze(0)
        ).item()
        
        # Label 정의: match_id는 같고 side(a/b)가 다른 경우만 Positive(1)
        name_i_id, name_j_id = name_i[:3], name_j[:3]
        name_i_side, name_j_side = name_i[-1], name_j[-1]
        
        is_positive = 1 if (name_i_id == name_j_id and name_i_side != name_j_side) else 0
        # 전체 지표용 (AUC, AP)
        all_scores.append(sim)
        all_labels.append(is_positive)
        
        # 쿼리별 지표용 (Recall@K)
        query_scores.append(sim)
        query_labels.append(is_positive)
    
    # 2. Recall@K 계산 (현재 쿼리 name_i에 대하여)
    query_scores = np.array(query_scores)
    query_labels = np.array(query_labels)
    
    # 점수 높은 순서대로 정렬 인덱스 추출
    sorted_indices = np.argsort(query_scores)[::-1]
    
    # print(recall_results)
    for k in ks:
        # 상위 K개 안에 정답(label=1)이 포함되어 있는지 확인
        if np.any(query_labels[sorted_indices[:k]] == 1):
            recall_results[k].append(1)
        else:
            recall_results[k].append(0)
    
    if np.argmax(query_labels) == np.argmax(query_scores):
        ox_counts +=1

# 3. 최종 지표 산출
all_scores = np.array(all_scores)
all_labels = np.array(all_labels)

auc = roc_auc_score(all_labels, all_scores)
ap  = average_precision_score(all_labels, all_scores)

print(f'AUC-ROC : {auc:.4f}')
print(f'AP      : {ap:.4f}')
for k in ks:
    r_at_k = np.mean(recall_results[k])
    print(f'Recall@{k}: {r_at_k:.4f}')

AUC-ROC : 0.9570
AP      : 0.4848
Recall@1: 0.7424
Recall@5: 0.8788
Recall@10: 0.8939


In [ ]:
# 시각화
results = []
for mid in test_ids:
    for query, target in [(f'{mid}_a', f'{mid}_b'),
                            (f'{mid}_b', f'{mid}_a')]:
        sims = {}
        for name in test_frames:
            if name == query:
                continue
            sims[name] = F.cosine_similarity(
                test_emb[query].unsqueeze(0),
                test_emb[name].unsqueeze(0)
            ).item()

        ranked     = sorted(sims.items(), key=lambda x: x[1], reverse=True)
        top_names  = [name  for name, _     in ranked[:5]]
        top_scores = [score for _,    score in ranked[:5]]

        results.append({
            'query':      query,
            'target':     target,
            'top_names':  top_names,
            'top_scores': top_scores,
            'hit_at_1':   target == top_names[0],
            'hit_at_5':   target in top_names[:5],
        })

success = [r for r in results if r['hit_at_5']]
failure = [r for r in results if not r['hit_at_5']]

print(f'Recall@5 Success: {len(success)}/{len(results)}')
print(f'Recall@5 Failure: {len(failure)}/{len(results)}')

for title, cases in [('Recall@5 - Success Cases', success[:]),
                        ('Recall@5 - Failure Cases', failure[:])]:
    if not cases:
        continue

    fig, axes = plt.subplots(
        len(cases), 6,
        figsize=(18, 3*len(cases)),
        layout='constrained'
    )
    fig.suptitle(title, fontsize=14)

    if len(cases) == 1:
        axes = [axes]

    for row, r in enumerate(cases):
        q_id, q_side = r['query'].rsplit('_', 1)
        q_img_path   = augmented_img_dir / f'{q_id}_{q_side}.jpg'

        ax = axes[row][0]
        if q_img_path:
            ax.imshow(mpimg.imread(q_img_path))
        else:
            ax.text(0.5, 0.5, 'No Image', ha='center', va='center')
        ax.set_title(f"Query\n{r['query']}\n", fontsize=10, fontweight='bold')
        ax.axis('off')

        for col, (name, score) in enumerate(
            zip(r['top_names'], r['top_scores'])
        ):
            n_id, n_side = name.rsplit('_', 1)
            n_img_path   = augmented_img_dir / f'{n_id}_{n_side}.jpg'
            is_target    = name == r['target']

            ax = axes[row][col+1]
            if n_img_path:
                ax.imshow(mpimg.imread(n_img_path))
            else:
                ax.text(0.5, 0.5, 'No Image', ha='center', va='center')

            label = f"{'[GT] ' if is_target else ''}Rank {col+1}\n{name}\n{score:.3f}"
            color = 'green' if is_target else 'black'
            weight = 'bold' if is_target else 'normal'
            ax.set_title(label, fontsize=10, color=color, fontweight=weight)

            for spine in ax.spines.values():
                spine.set_edgecolor('green' if is_target else 'gray')
                spine.set_linewidth(3 if is_target else 0.5)
            ax.axis('off')

    save_path = f'../0423/{title.replace(" ", "_").replace("-", "_")}.jpg'
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path}')